In [1]:
import os

In [2]:
%pwd

'c:\\Users\\SAI\\Desktop\\data science\\Text-Summarization-Project\\research'

In [18]:
import os
from pathlib import Path

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "config" / "config.yaml").exists():
    project_root = project_root.parent

if not (project_root / "config" / "config.yaml").exists():
    raise FileNotFoundError("Could not locate the project root containing config/config.yaml")

os.chdir(project_root)
print(f"Working directory: {Path.cwd()}")

Working directory: c:\Users\SAI\Desktop\data science\Text-Summarization-Project


In [4]:
%pwd

'c:\\Users\\SAI\\Desktop\\data science\\Text-Summarization-Project'

In [26]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [27]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [28]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
           
        )

        return model_evaluation_config

In [29]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
from rouge_score import rouge_scorer, scoring
import torch
import pandas as pd
from tqdm import tqdm

In [30]:
class ModelEvaluation:
    def __init__(self, config: "ModelEvaluationConfig"):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """Split a dataset column into batches for model inference."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=4,
        device="cuda" if torch.cuda.is_available() else "cpu",
        column_text="article",
        column_summary="highlights",
        max_input_length=256,
        max_output_length=32,
        num_beams=1,
    ):
        article_batches = self.generate_batch_sized_chunks(dataset[column_text], batch_size)
        target_batches = self.generate_batch_sized_chunks(dataset[column_summary], batch_size)
        aggregator = scoring.BootstrapAggregator()

        model.eval()
        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=(len(dataset[column_text]) + batch_size - 1) // batch_size,
        ):
            inputs = tokenizer(
                article_batch,
                max_length=max_input_length,
                truncation=True,
                padding=True,
                return_tensors="pt",
            )

            with torch.no_grad():
                summaries = model.generate(
                    input_ids=inputs["input_ids"].to(device),
                    attention_mask=inputs["attention_mask"].to(device),
                    num_beams=num_beams,
                    max_length=max_output_length,
                    do_sample=False,
                )

            decoded_summaries = [
                tokenizer.decode(
                    summary,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                ).replace("<n>", " ")
                for summary in summaries
            ]

            for prediction, reference in zip(decoded_summaries, target_batch):
                aggregator.add_scores(metric.score(reference, prediction))

        return aggregator.aggregate()

    def evaluate(self, sample_count=1):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rouge_metric = rouge_scorer.RougeScorer(rouge_names, use_stemmer=True)

        score = self.calculate_metric_on_test_ds(
            dataset_samsum_pt["test"][:sample_count],
            rouge_metric,
            model_pegasus,
            tokenizer,
            batch_size=4,
            column_text="dialogue",
            column_summary="summary",
        )

        rouge_dict = {name: score[name].mid.fmeasure for name in rouge_names}
        pd.DataFrame(rouge_dict, index=["pegasus"]).to_csv(
            self.config.metric_file_name,
            index=False,
        )
        return rouge_dict

In [25]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-09-07 04:54:16,896: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-07 04:54:16,898: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-07 04:54:16,900: INFO: common: created directory at: artifacts]
[2026-09-07 04:54:16,903: INFO: common: created directory at: artifacts/model_evaluation]


Loading weights: 100%|██████████| 680/680 [00:00<00:00, 4080.50it/s]


[2026-09-07 04:54:22,491: INFO: rouge_scorer: Using default tokenizer.]


100%|██████████| 1/1 [00:05<00:00,  5.53s/it]
